# CHB-MIT · Event scorer and protocol tests
**Next stage: fix the evaluation rules before full training.**

Run **Sections 1–5** on CPU. These sections use artificial examples with known answers. No model training, real performance evaluation, or preprocessing is repeated. Section 5 saves a protocol specification and test results in your existing run.

This notebook implements a proposed, explicit research scoring policy. It does not establish clinical acceptability. Its default threshold of 0.5 is **only for synthetic tests**. An actual threshold must later be selected on development validation predictions, under a declared selection objective; the held-out test subject must not select it.

Your eight-prediction smoke preview is too short for performance scoring. The full-export validator deliberately rejects it.

## 1 · Dependencies and Drive

In [ ]:
%pip -q install numpy pandas h5py
from pathlib import Path
import json
from google.colab import drive
drive.mount('/content/gdrive')
RUN=Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
assert (RUN/'locked_folds.json').exists(), 'Check the run path and Drive mount.'

## 2 · Frozen scoring conventions

| Item | Rule |
|---|---|
| Forecast interval | Alarm at t predicts onset in **(t+300, t+2100]** seconds: open lower, closed upper boundary |
| Trigger | A ready causal score ≥ threshold, when no cooldown is active |
| Repeat alarms | No new alarm before t+2100; reissue at expiry if score is still high; no rising-edge requirement |
| Ictal/postictal gating | None in this version; training-label exclusions do not suppress replay alarms |
| Gaps | No predictions during unavailable intervals; an existing alarm/cooldown continues in wall-clock time across gaps/files in the same case |
| Main sensitivity | Unique eligible targets predicted / **all** eligible targets in the evaluated role, including those without forecast opportunity |
| Opportunity-conditioned sensitivity | Separate secondary metric using targets with at least one ready decision in their valid warning interval |
| No eligible targets | Sensitivity undefined, not zero; retain the case for alarm and coverage results |
| False alarm | No annotated seizure onset of any kind in the SOP, with fully observed future annotations through SPH+SOP |
| Non-target seizure | An SOP containing only non-target seizure onset(s) is reported separately; it is not a target success |
| Early seizure | Onset in (t,t+SPH] is flagged; it cannot make the target a success and does not excuse an otherwise false alarm |
| Unknown outcome | No observed seizure in the SOP and future annotations incomplete: unscorable, not false |
| Partial observed positive | An observed target/non-target onset can establish an association despite later annotation gaps; report the incomplete future separately |

**Exposure convention:** count one four-second decision bin per stored window. Missing windows are not interpolated. Prediction-ready hours use ready bins. *Evaluable decision hours* use ready bins whose following 35 minutes have continuous annotation coverage, even if signal quality later becomes invalid. False alarms per hour use this latter denominator. It includes ictal and postictal time, so label it explicitly; it is not interchangeable with an interictal-only FPR/h.

Time in warning is the fraction of evaluable (and separately ready) decision bins at which a 35-minute alert is active. The five-minute SPH is included in alert burden. This is a four-second discretization, not continuous-time interpolation. Cooldown time remains in denominators. Report total alarm burden and non-target-associated alarms alongside false alarms to avoid hiding burden.

Each case uses its own time origin. Aggregate sensitivity by summing target counts, and report mean case sensitivity separately. In the current cohort each included case corresponds to one subject; repeated-subject cases would require subject-level aggregation.

The frozen 47-target cohort remains intact across all outer tests. A single inner split has fewer targets because it excludes the held-out subject.

In [ ]:
"""Fixed research scoring protocol v1. No training or threshold optimization."""
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd

@dataclass(frozen=True)
class ScoringPolicy:
    sph_s: float = 300.0
    sop_s: float = 1800.0
    decision_bin_s: float = 4.0
    threshold: float = 0.5  # Synthetic demonstration only; not a selected operating point.

    def __post_init__(self):
        if self.sph_s<0 or self.sop_s<=0 or self.decision_bin_s<=0 or not 0<=self.threshold<=1:
            raise ValueError('Invalid scoring policy')


def merge_intervals(intervals):
    out=[]
    for a,b in sorted((float(a),float(b)) for a,b in intervals):
        if not np.isfinite([a,b]).all() or b<=a: raise ValueError('Invalid coverage interval')
        if out and a<=out[-1][1]:out[-1]=(out[-1][0],max(out[-1][1],b))
        else:out.append((a,b))
    return out


def fully_covered(a,b,intervals):
    return any(x<=a and y>=b for x,y in intervals)


def score_case(predictions,events,coverage,policy=ScoringPolicy()):
    """One case, one relative-time origin. Future annotations label outcomes only.

    predictions: ALL stored decision rows, including unavailable rows. Required:
    decision_s, prediction_ready(bool), score(float, NaN when unavailable).
    events: event_id, onset_s, offset_s, eligible_target(bool).
    coverage: annotation-known intervals, including montage-unusable recordings.
    Exposure uses discrete decision bins, not interpolation over signal gaps.
    """
    p=predictions.copy();e=events.copy();obs=merge_intervals(coverage)
    required={'decision_s','prediction_ready','score'}
    if not required.issubset(p):raise ValueError(f'Missing prediction fields: {required-set(p.columns)}')
    if not {'event_id','onset_s','offset_s','eligible_target'}.issubset(e):raise ValueError('Missing event fields')
    if not p.prediction_ready.isin([True,False]).all() or not e.eligible_target.isin([True,False]).all():
        raise ValueError('Availability and eligibility must be boolean')
    p=p.sort_values('decision_s').reset_index(drop=True)
    t=p.decision_s.to_numpy(dtype=float);ready=p.prediction_ready.to_numpy(dtype=bool)
    scores=p.score.to_numpy(dtype=float)
    if not np.isfinite(t).all() or (len(t)>1 and np.any(np.diff(t)<policy.decision_bin_s-1e-6)):
        raise ValueError('Duplicate, overlapping, or invalid decision times')
    if not np.isfinite(scores[ready]).all() or np.any((scores[ready]<0)|(scores[ready]>1)):
        raise ValueError('Ready scores must be finite probabilities in [0,1]')
    if np.any(np.isfinite(scores[~ready])):raise ValueError('Unavailable scores must be NaN')
    if e.event_id.duplicated().any() or not np.isfinite(e[['onset_s','offset_s']].to_numpy(dtype=float)).all() or np.any(e.offset_s<=e.onset_s):
        raise ValueError('Invalid event table')
    for x in t:
        if not fully_covered(x-policy.decision_bin_s,x,obs):
            raise ValueError('Decision bin is outside annotated recording coverage')
    for row in e.itertuples():
        if not fully_covered(row.onset_s,row.offset_s,obs):
            raise ValueError('Event lies outside known annotations')
    H=policy.sph_s+policy.sop_s
    evaluable=np.array([fully_covered(x,x+H,obs) for x in t],dtype=bool)&ready
    alarms=[];next_allowed=-np.inf
    # Alarm generation uses scores, readiness, time and fixed cooldown only.
    # It never reads eligibility, seizure labels or future coverage.
    for i,x in enumerate(t):
        if not ready[i] or scores[i]<policy.threshold or x<next_allowed:continue
        alarm_id=len(alarms);a=x+policy.sph_s;b=x+H
        within=e[(e.onset_s>a)&(e.onset_s<=b)]
        target_ids=within.loc[within.eligible_target.astype(bool),'event_id'].tolist()
        non_target_ids=within.loc[~within.eligible_target.astype(bool),'event_id'].tolist()
        known=fully_covered(x,b,obs)
        outcome=('target_hit' if target_ids else 'non_target_seizure' if non_target_ids else
                 'false_alarm' if known else 'unscorable')
        alarms.append(dict(alarm_id=alarm_id,alarm_s=x,sop_start_exclusive_s=a,sop_end_inclusive_s=b,
                           threshold=policy.threshold,score=float(scores[i]),outcome=outcome,
                           future_annotation_complete=known,origin_evaluable=bool(evaluable[i]),
                           early_seizure=bool(((e.onset_s>x)&(e.onset_s<=a)).any()),
                           target_ids=target_ids,non_target_ids=non_target_ids))
        next_allowed=b  # 35 minutes, including SPH, independent of observed outcomes.
    alarm_columns=['alarm_id','alarm_s','sop_start_exclusive_s','sop_end_inclusive_s','threshold','score','outcome',
                   'future_annotation_complete','origin_evaluable','early_seizure','target_ids','non_target_ids']
    alarm_table=pd.DataFrame(alarms,columns=alarm_columns)
    targets=[]
    for row in e[e.eligible_target.astype(bool)].itertuples():
        opportunities=ready&(t>=row.onset_s-H)&(t<row.onset_s-policy.sph_s)
        matches=[a for a in alarms if row.event_id in a['target_ids']]
        targets.append(dict(event_id=row.event_id,onset_s=float(row.onset_s),retained=True,
                            forecast_opportunities=int(opportunities.sum()),has_opportunity=bool(opportunities.any()),
                            predicted=bool(matches),matched_alarm_ids=[a['alarm_id'] for a in matches],
                            lead_time_s=float(row.onset_s-matches[0]['alarm_s']) if matches else np.nan))
    target_table=pd.DataFrame(targets,columns=['event_id','onset_s','retained','forecast_opportunities','has_opportunity','predicted','matched_alarm_ids','lead_time_s'])
    # Decision-bin burden: warning is active at decision t when alarm <= t < alarm+H.
    active=np.zeros(len(t),dtype=bool)
    for a in alarms:active|=(t>=a['alarm_s'])&(t<a['sop_end_inclusive_s'])
    hits=sum(x['predicted'] for x in targets);opp=sum(x['has_opportunity'] for x in targets)
    opp_hits=sum(x['predicted'] and x['has_opportunity'] for x in targets)
    false=sum(a['outcome']=='false_alarm' for a in alarms)
    nt=sum(a['outcome']=='non_target_seizure' for a in alarms)
    eval_nt=sum(a['outcome']=='non_target_seizure' and a['origin_evaluable'] for a in alarms)
    ready_h=float(ready.sum()*policy.decision_bin_s/3600)
    eval_h=float(evaluable.sum()*policy.decision_bin_s/3600)
    summary=dict(targets=len(targets),predicted_targets=hits,targets_with_opportunity=opp,
                 sensitivity=hits/len(targets) if targets else None,
                 opportunity_conditioned_sensitivity=opp_hits/opp if opp else None,
                 recorded_decision_hours=len(t)*policy.decision_bin_s/3600,
                 prediction_ready_hours=ready_h,evaluable_decision_hours=eval_h,
                 prediction_availability=float(ready.mean()) if len(t) else None,
                 alarm_count=len(alarms),false_alarms=false,non_target_seizure_alarms=nt,
                 unscorable_alarms=sum(a['outcome']=='unscorable' for a in alarms),
                 incomplete_future_alarms=sum(not a['future_annotation_complete'] for a in alarms),
                 false_alarms_per_evaluable_hour=false/eval_h if eval_h else None,
                 non_target_alarms_at_evaluable_origins=eval_nt,
                 alarms_without_target_hit_per_evaluable_hour=(false+eval_nt)/eval_h if eval_h else None,
                 alarms_per_prediction_ready_hour=len(alarms)/ready_h if ready_h else None,
                 warning_evaluable_bins=int((active&evaluable).sum()),evaluable_bins=int(evaluable.sum()),
                 warning_ready_bins=int((active&ready).sum()),ready_bins=int(ready.sum()),
                 time_in_warning_evaluable=float((active&evaluable).sum()/evaluable.sum()) if evaluable.any() else None,
                 time_in_warning_ready=float((active&ready).sum()/ready.sum()) if ready.any() else None)
    return summary,alarm_table,target_table


def score_cohort(predictions,events,coverage_by_case,policy=ScoringPolicy()):
    if 'case' not in predictions or 'case' not in events:raise ValueError('Case IDs required')
    unknown=(set(predictions['case'])|set(events['case']))-set(coverage_by_case)
    if unknown:raise ValueError(f'Missing coverage: {unknown}')
    summaries=[];alarms=[];targets=[]
    for case,coverage in sorted(coverage_by_case.items()):
        s,a,e=score_case(predictions[predictions['case']==case],events[events['case']==case],coverage,policy)
        summaries.append(dict(case=case,**s));a.insert(0,'case',case);e.insert(0,'case',case)
        alarms.append(a);targets.append(e)
    table=pd.DataFrame(summaries)
    def total(name):return float(table[name].sum())
    def ratio(a,b):return total(a)/total(b) if total(b) else None
    eligible=table[table.targets>0]
    pooled=dict(targets=int(total('targets')),predicted_targets=int(total('predicted_targets')),
                sensitivity=ratio('predicted_targets','targets'),
                macro_case_sensitivity=float(eligible.sensitivity.mean()) if len(eligible) else None,
                false_alarms_per_evaluable_hour=ratio('false_alarms','evaluable_decision_hours'),
                time_in_warning_evaluable=ratio('warning_evaluable_bins','evaluable_bins'),
                prediction_availability=ratio('prediction_ready_hours','recorded_decision_hours'),
                unscorable_alarms=int(total('unscorable_alarms')),non_target_seizure_alarms=int(total('non_target_seizure_alarms')))
    return pooled,table,pd.concat(alarms,ignore_index=True),pd.concat(targets,ignore_index=True)


def validate_full_prediction_export(data, role, predictions, provenance):
    """Use with FoldData from the loader notebook, before score_cohort.
    Reject incomplete previews and mismatched roles/scalers. No signal reads.
    """
    import hashlib
    if provenance.get('purpose')!='full_chronological_predictions' or provenance.get('smoke_only',True):
        raise ValueError('Smoke-test previews cannot be scored as full predictions')
    expected={'outer':data.outer,'scope':data.scope,'role':role,
              'scaler_sha256':hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),
              'folds_sha256':hashlib.sha256((data.run/'locked_folds.json').read_bytes()).hexdigest()}
    for key,value in expected.items():
        if provenance.get(key)!=value:raise ValueError('Prediction provenance mismatch: '+key)
    if not isinstance(provenance.get('model_checkpoint_sha256'),str) or len(provenance['model_checkpoint_sha256'])!=64:
        raise ValueError('Model checkpoint hash required')
    required={'relative_path','index','case','decision_s','prediction_ready','score'}
    if not required.issubset(predictions):raise ValueError('Incomplete prediction schema')
    p=predictions.copy()
    if p[['relative_path','index']].duplicated().any():raise ValueError('Duplicate prediction keys')
    rows=data.records(role);expected_paths={r['relative_path'] for r in rows}
    if set(p.relative_path)!=expected_paths:raise ValueError('Missing or unexpected recordings')
    for row in rows:
        m=data.metadata(row);part=p[p.relative_path==row['relative_path']].sort_values('index')
        n=len(m['decision_s'])
        if len(part)!=n or not np.array_equal(part['index'].to_numpy(),np.arange(n)):
            raise ValueError('Missing windows: '+row['relative_path'])
        if not (part['case']==row['case']).all() or not np.allclose(part.decision_s,m['decision_s'],rtol=0,atol=1e-6):
            raise ValueError('Case/timestamps mismatch')
        if not part.prediction_ready.isin([True,False]).all() or not np.array_equal(part.prediction_ready.to_numpy(dtype=bool),m['prediction_ready']):
            raise ValueError('Prediction availability differs from locked loader policy')
    # Include annotated recordings with unusable montage in outcome coverage.
    # Prediction availability and annotation availability are distinct concepts.
    allowed=set(data.roles[role])
    allrows=[r for r in data.inventory if r['subject'] in allowed and not r['excluded_case']]
    cases={r['case'] for r in allrows}
    coverage={case:[(r['start_s'],r['end_s']) for r in allrows if r['case']==case] for case in cases}
    events=pd.DataFrame([e for e in data.events if e['case'] in cases])
    return p,events,coverage


## 3 · Run known-answer tests
Checks interval boundaries, cooldown, annotation gaps, missed targets, censored alarms, zero-target cases, outcome-independent triggering, and cohort aggregation.

In [ ]:
import numpy as np
import pandas as pd

def run_scoring_tests():
    passed=[]
    def events(rows):return pd.DataFrame(rows,columns=['event_id','onset_s','offset_s','eligible_target'])
    def predictions(times,high=(),ready=None):
        t=np.array(times,dtype=float);r=np.ones(len(t),dtype=bool) if ready is None else np.array(ready,dtype=bool)
        return pd.DataFrame(dict(decision_s=t,prediction_ready=r,score=[(0.9 if x in high else 0.1) if ok else np.nan for x,ok in zip(t,r)]))
    p=predictions(range(4,6001,4),high=[4,8,2104,4204])
    e=events([['target',1600,1604,True],['too_early',2404,2408,True]])
    s,a,t=score_case(p,e,[(0,6000)])
    assert a.alarm_s.tolist()==[4,2104,4204]
    assert a.outcome.tolist()==['target_hit','false_alarm','unscorable']
    assert s['sensitivity']==0.5 and s['false_alarms']==1 and s['unscorable_alarms']==1
    assert s['evaluable_bins']==975 and np.isclose(s['false_alarms_per_evaluable_hour'],3600/3900)
    passed.append('cooldown, early seizure, censored alarm and exact denominator')
    # Both SPH/SOP boundaries and target matching, independent of forecast labels.
    p=predictions([4],high=[4]);e=events([['lower',304,308,True],['upper',2104,2108,True]])
    s,a,t=score_case(p,e,[(0,6000)])
    assert t.predicted.tolist()==[False,True] and t.has_opportunity.tolist()==[False,True]
    passed.append('open SPH boundary, closed SOP boundary')
    # Gaps must not be mistaken for seizure-free time or reset live alarms.
    p=predictions([4,1004,2104],high=[4,1004,2104])
    s,a,t=score_case(p,events([]),[(0,500),(1000,6000)])
    assert a.alarm_s.tolist()==[4,2104] and a.outcome.tolist()==['unscorable','false_alarm']
    passed.append('annotation gaps and cooldown persistence across gaps')
    p=predictions([4],high=[4]);e=events([['nt',1600,1604,False]])
    s,a,t=score_case(p,e,[(0,6000)])
    assert s['non_target_seizure_alarms']==1 and s['false_alarms']==0 and s['sensitivity'] is None
    assert s['alarms_without_target_hit_per_evaluable_hour']==900
    passed.append('non-target seizures separately reported, zero-target sensitivity undefined')
    # Known positive can be matched despite missing later annotations, but is flagged.
    s,a,t=score_case(p,events([['hit',400,404,True]]),[(0,500)])
    assert s['sensitivity']==1 and s['incomplete_future_alarms']==1 and s['evaluable_decision_hours']==0
    passed.append('observed hit with censored future retained and flagged')
    s,a,t=score_case(predictions([4,8],ready=[False,False]),events([['miss',400,404,True]]),[(0,6000)])
    assert s['sensitivity']==0 and s['targets_with_opportunity']==0 and s['opportunity_conditioned_sensitivity'] is None
    passed.append('unavailable input does not remove target from denominator')
    # Alarm times must not depend on annotations or future observation coverage.
    p=predictions([4,8,2104],high=[4,8,2104])
    _,a,_=score_case(p,events([]),[(0,6000)])
    _,b,_=score_case(p,events([['change',400,404,True]]),[(0,6000)])
    assert a.alarm_s.tolist()==b.alarm_s.tolist()
    passed.append('causal alarm generation independent of seizure outcomes')
    bad=predictions([4,4])
    try:score_case(bad,events([]),[(0,6000)]);raise AssertionError('duplicates accepted')
    except ValueError:pass
    bad=predictions([4],ready=[False]);bad['score']=0.9
    try:score_case(bad,events([]),[(0,6000)]);raise AssertionError('unavailable finite score accepted')
    except ValueError:pass
    passed.append('duplicate timestamps and unavailable finite scores rejected')
    # Same case-relative times in distinct patients must never be mixed.
    p1=predictions([4],high=[4]).assign(case='A');p2=predictions([4]).assign(case='B')
    e1=events([['a',400,404,True]]).assign(case='A');e2=events([['b',400,404,True]]).assign(case='B')
    pooled,cases,_,_=score_cohort(pd.concat([p1,p2]),pd.concat([e1,e2]),{'A':[(0,6000)],'B':[(0,6000)]})
    assert pooled['targets']==2 and pooled['predicted_targets']==1 and pooled['sensitivity']==0.5
    passed.append('case-relative timelines and pooled target counts remain separate')
    return passed


passed_tests=run_scoring_tests()
for name in passed_tests: print('PASS:',name)
print('All',len(passed_tests),'test groups passed.')

## 4 · Inspect a worked example
A synthetic case has two targets and sustained above-threshold scores at selected times. Expected: alarms at 4, 2104, and 4204 seconds; one target hit, one false alarm (a seizure arrives exactly at the excluded five-minute boundary), and one unscorable alarm near the recording end. Sensitivity is 1/2. These numbers are artificial.

In [ ]:
synthetic=pd.DataFrame({'decision_s':np.arange(4,6001,4,dtype=float)})
synthetic['prediction_ready']=True
synthetic['score']=np.where(synthetic.decision_s.isin([4,8,2104,4204]),0.9,0.1)
synthetic_events=pd.DataFrame([
    dict(event_id='synthetic_target_1',onset_s=1600.,offset_s=1604.,eligible_target=True),
    dict(event_id='synthetic_target_2',onset_s=2404.,offset_s=2408.,eligible_target=True)])
synthetic_summary,synthetic_alarms,synthetic_targets=score_case(synthetic,synthetic_events,[(0,6000)])
display(synthetic_alarms)
display(synthetic_targets)
print(json.dumps(synthetic_summary,indent=2))

## 5 · Save the protocol and test results
This saves no real EEG performance claim. It records the exact scorer source hash and exposes threshold selection as unfinished. Reports use a fresh timestamped folder.

In [ ]:
SCORER_SOURCE = '"""Fixed research scoring protocol v1. No training or threshold optimization."""\nfrom dataclasses import dataclass, asdict\nimport numpy as np\nimport pandas as pd\n\n@dataclass(frozen=True)\nclass ScoringPolicy:\n    sph_s: float = 300.0\n    sop_s: float = 1800.0\n    decision_bin_s: float = 4.0\n    threshold: float = 0.5  # Synthetic demonstration only; not a selected operating point.\n\n    def __post_init__(self):\n        if self.sph_s<0 or self.sop_s<=0 or self.decision_bin_s<=0 or not 0<=self.threshold<=1:\n            raise ValueError(\'Invalid scoring policy\')\n\n\ndef merge_intervals(intervals):\n    out=[]\n    for a,b in sorted((float(a),float(b)) for a,b in intervals):\n        if not np.isfinite([a,b]).all() or b<=a: raise ValueError(\'Invalid coverage interval\')\n        if out and a<=out[-1][1]:out[-1]=(out[-1][0],max(out[-1][1],b))\n        else:out.append((a,b))\n    return out\n\n\ndef fully_covered(a,b,intervals):\n    return any(x<=a and y>=b for x,y in intervals)\n\n\ndef score_case(predictions,events,coverage,policy=ScoringPolicy()):\n    """One case, one relative-time origin. Future annotations label outcomes only.\n\n    predictions: ALL stored decision rows, including unavailable rows. Required:\n    decision_s, prediction_ready(bool), score(float, NaN when unavailable).\n    events: event_id, onset_s, offset_s, eligible_target(bool).\n    coverage: annotation-known intervals, including montage-unusable recordings.\n    Exposure uses discrete decision bins, not interpolation over signal gaps.\n    """\n    p=predictions.copy();e=events.copy();obs=merge_intervals(coverage)\n    required={\'decision_s\',\'prediction_ready\',\'score\'}\n    if not required.issubset(p):raise ValueError(f\'Missing prediction fields: {required-set(p.columns)}\')\n    if not {\'event_id\',\'onset_s\',\'offset_s\',\'eligible_target\'}.issubset(e):raise ValueError(\'Missing event fields\')\n    if not p.prediction_ready.isin([True,False]).all() or not e.eligible_target.isin([True,False]).all():\n        raise ValueError(\'Availability and eligibility must be boolean\')\n    p=p.sort_values(\'decision_s\').reset_index(drop=True)\n    t=p.decision_s.to_numpy(dtype=float);ready=p.prediction_ready.to_numpy(dtype=bool)\n    scores=p.score.to_numpy(dtype=float)\n    if not np.isfinite(t).all() or (len(t)>1 and np.any(np.diff(t)<policy.decision_bin_s-1e-6)):\n        raise ValueError(\'Duplicate, overlapping, or invalid decision times\')\n    if not np.isfinite(scores[ready]).all() or np.any((scores[ready]<0)|(scores[ready]>1)):\n        raise ValueError(\'Ready scores must be finite probabilities in [0,1]\')\n    if np.any(np.isfinite(scores[~ready])):raise ValueError(\'Unavailable scores must be NaN\')\n    if e.event_id.duplicated().any() or not np.isfinite(e[[\'onset_s\',\'offset_s\']].to_numpy(dtype=float)).all() or np.any(e.offset_s<=e.onset_s):\n        raise ValueError(\'Invalid event table\')\n    for x in t:\n        if not fully_covered(x-policy.decision_bin_s,x,obs):\n            raise ValueError(\'Decision bin is outside annotated recording coverage\')\n    for row in e.itertuples():\n        if not fully_covered(row.onset_s,row.offset_s,obs):\n            raise ValueError(\'Event lies outside known annotations\')\n    H=policy.sph_s+policy.sop_s\n    evaluable=np.array([fully_covered(x,x+H,obs) for x in t],dtype=bool)&ready\n    alarms=[];next_allowed=-np.inf\n    # Alarm generation uses scores, readiness, time and fixed cooldown only.\n    # It never reads eligibility, seizure labels or future coverage.\n    for i,x in enumerate(t):\n        if not ready[i] or scores[i]<policy.threshold or x<next_allowed:continue\n        alarm_id=len(alarms);a=x+policy.sph_s;b=x+H\n        within=e[(e.onset_s>a)&(e.onset_s<=b)]\n        target_ids=within.loc[within.eligible_target.astype(bool),\'event_id\'].tolist()\n        non_target_ids=within.loc[~within.eligible_target.astype(bool),\'event_id\'].tolist()\n        known=fully_covered(x,b,obs)\n        outcome=(\'target_hit\' if target_ids else \'non_target_seizure\' if non_target_ids else\n                 \'false_alarm\' if known else \'unscorable\')\n        alarms.append(dict(alarm_id=alarm_id,alarm_s=x,sop_start_exclusive_s=a,sop_end_inclusive_s=b,\n                           threshold=policy.threshold,score=float(scores[i]),outcome=outcome,\n                           future_annotation_complete=known,origin_evaluable=bool(evaluable[i]),\n                           early_seizure=bool(((e.onset_s>x)&(e.onset_s<=a)).any()),\n                           target_ids=target_ids,non_target_ids=non_target_ids))\n        next_allowed=b  # 35 minutes, including SPH, independent of observed outcomes.\n    alarm_columns=[\'alarm_id\',\'alarm_s\',\'sop_start_exclusive_s\',\'sop_end_inclusive_s\',\'threshold\',\'score\',\'outcome\',\n                   \'future_annotation_complete\',\'origin_evaluable\',\'early_seizure\',\'target_ids\',\'non_target_ids\']\n    alarm_table=pd.DataFrame(alarms,columns=alarm_columns)\n    targets=[]\n    for row in e[e.eligible_target.astype(bool)].itertuples():\n        opportunities=ready&(t>=row.onset_s-H)&(t<row.onset_s-policy.sph_s)\n        matches=[a for a in alarms if row.event_id in a[\'target_ids\']]\n        targets.append(dict(event_id=row.event_id,onset_s=float(row.onset_s),retained=True,\n                            forecast_opportunities=int(opportunities.sum()),has_opportunity=bool(opportunities.any()),\n                            predicted=bool(matches),matched_alarm_ids=[a[\'alarm_id\'] for a in matches],\n                            lead_time_s=float(row.onset_s-matches[0][\'alarm_s\']) if matches else np.nan))\n    target_table=pd.DataFrame(targets,columns=[\'event_id\',\'onset_s\',\'retained\',\'forecast_opportunities\',\'has_opportunity\',\'predicted\',\'matched_alarm_ids\',\'lead_time_s\'])\n    # Decision-bin burden: warning is active at decision t when alarm <= t < alarm+H.\n    active=np.zeros(len(t),dtype=bool)\n    for a in alarms:active|=(t>=a[\'alarm_s\'])&(t<a[\'sop_end_inclusive_s\'])\n    hits=sum(x[\'predicted\'] for x in targets);opp=sum(x[\'has_opportunity\'] for x in targets)\n    opp_hits=sum(x[\'predicted\'] and x[\'has_opportunity\'] for x in targets)\n    false=sum(a[\'outcome\']==\'false_alarm\' for a in alarms)\n    nt=sum(a[\'outcome\']==\'non_target_seizure\' for a in alarms)\n    eval_nt=sum(a[\'outcome\']==\'non_target_seizure\' and a[\'origin_evaluable\'] for a in alarms)\n    ready_h=float(ready.sum()*policy.decision_bin_s/3600)\n    eval_h=float(evaluable.sum()*policy.decision_bin_s/3600)\n    summary=dict(targets=len(targets),predicted_targets=hits,targets_with_opportunity=opp,\n                 sensitivity=hits/len(targets) if targets else None,\n                 opportunity_conditioned_sensitivity=opp_hits/opp if opp else None,\n                 recorded_decision_hours=len(t)*policy.decision_bin_s/3600,\n                 prediction_ready_hours=ready_h,evaluable_decision_hours=eval_h,\n                 prediction_availability=float(ready.mean()) if len(t) else None,\n                 alarm_count=len(alarms),false_alarms=false,non_target_seizure_alarms=nt,\n                 unscorable_alarms=sum(a[\'outcome\']==\'unscorable\' for a in alarms),\n                 incomplete_future_alarms=sum(not a[\'future_annotation_complete\'] for a in alarms),\n                 false_alarms_per_evaluable_hour=false/eval_h if eval_h else None,\n                 non_target_alarms_at_evaluable_origins=eval_nt,\n                 alarms_without_target_hit_per_evaluable_hour=(false+eval_nt)/eval_h if eval_h else None,\n                 alarms_per_prediction_ready_hour=len(alarms)/ready_h if ready_h else None,\n                 warning_evaluable_bins=int((active&evaluable).sum()),evaluable_bins=int(evaluable.sum()),\n                 warning_ready_bins=int((active&ready).sum()),ready_bins=int(ready.sum()),\n                 time_in_warning_evaluable=float((active&evaluable).sum()/evaluable.sum()) if evaluable.any() else None,\n                 time_in_warning_ready=float((active&ready).sum()/ready.sum()) if ready.any() else None)\n    return summary,alarm_table,target_table\n\n\ndef score_cohort(predictions,events,coverage_by_case,policy=ScoringPolicy()):\n    if \'case\' not in predictions or \'case\' not in events:raise ValueError(\'Case IDs required\')\n    unknown=(set(predictions[\'case\'])|set(events[\'case\']))-set(coverage_by_case)\n    if unknown:raise ValueError(f\'Missing coverage: {unknown}\')\n    summaries=[];alarms=[];targets=[]\n    for case,coverage in sorted(coverage_by_case.items()):\n        s,a,e=score_case(predictions[predictions[\'case\']==case],events[events[\'case\']==case],coverage,policy)\n        summaries.append(dict(case=case,**s));a.insert(0,\'case\',case);e.insert(0,\'case\',case)\n        alarms.append(a);targets.append(e)\n    table=pd.DataFrame(summaries)\n    def total(name):return float(table[name].sum())\n    def ratio(a,b):return total(a)/total(b) if total(b) else None\n    eligible=table[table.targets>0]\n    pooled=dict(targets=int(total(\'targets\')),predicted_targets=int(total(\'predicted_targets\')),\n                sensitivity=ratio(\'predicted_targets\',\'targets\'),\n                macro_case_sensitivity=float(eligible.sensitivity.mean()) if len(eligible) else None,\n                false_alarms_per_evaluable_hour=ratio(\'false_alarms\',\'evaluable_decision_hours\'),\n                time_in_warning_evaluable=ratio(\'warning_evaluable_bins\',\'evaluable_bins\'),\n                prediction_availability=ratio(\'prediction_ready_hours\',\'recorded_decision_hours\'),\n                unscorable_alarms=int(total(\'unscorable_alarms\')),non_target_seizure_alarms=int(total(\'non_target_seizure_alarms\')))\n    return pooled,table,pd.concat(alarms,ignore_index=True),pd.concat(targets,ignore_index=True)\n\n\ndef validate_full_prediction_export(data, role, predictions, provenance):\n    """Use with FoldData from the loader notebook, before score_cohort.\n    Reject incomplete previews and mismatched roles/scalers. No signal reads.\n    """\n    import hashlib\n    if provenance.get(\'purpose\')!=\'full_chronological_predictions\' or provenance.get(\'smoke_only\',True):\n        raise ValueError(\'Smoke-test previews cannot be scored as full predictions\')\n    expected={\'outer\':data.outer,\'scope\':data.scope,\'role\':role,\n              \'scaler_sha256\':hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),\n              \'folds_sha256\':hashlib.sha256((data.run/\'locked_folds.json\').read_bytes()).hexdigest()}\n    for key,value in expected.items():\n        if provenance.get(key)!=value:raise ValueError(\'Prediction provenance mismatch: \'+key)\n    if not isinstance(provenance.get(\'model_checkpoint_sha256\'),str) or len(provenance[\'model_checkpoint_sha256\'])!=64:\n        raise ValueError(\'Model checkpoint hash required\')\n    required={\'relative_path\',\'index\',\'case\',\'decision_s\',\'prediction_ready\',\'score\'}\n    if not required.issubset(predictions):raise ValueError(\'Incomplete prediction schema\')\n    p=predictions.copy()\n    if p[[\'relative_path\',\'index\']].duplicated().any():raise ValueError(\'Duplicate prediction keys\')\n    rows=data.records(role);expected_paths={r[\'relative_path\'] for r in rows}\n    if set(p.relative_path)!=expected_paths:raise ValueError(\'Missing or unexpected recordings\')\n    for row in rows:\n        m=data.metadata(row);part=p[p.relative_path==row[\'relative_path\']].sort_values(\'index\')\n        n=len(m[\'decision_s\'])\n        if len(part)!=n or not np.array_equal(part[\'index\'].to_numpy(),np.arange(n)):\n            raise ValueError(\'Missing windows: \'+row[\'relative_path\'])\n        if not (part[\'case\']==row[\'case\']).all() or not np.allclose(part.decision_s,m[\'decision_s\'],rtol=0,atol=1e-6):\n            raise ValueError(\'Case/timestamps mismatch\')\n        if not part.prediction_ready.isin([True,False]).all() or not np.array_equal(part.prediction_ready.to_numpy(dtype=bool),m[\'prediction_ready\']):\n            raise ValueError(\'Prediction availability differs from locked loader policy\')\n    # Include annotated recordings with unusable montage in outcome coverage.\n    # Prediction availability and annotation availability are distinct concepts.\n    allowed=set(data.roles[role])\n    allrows=[r for r in data.inventory if r[\'subject\'] in allowed and not r[\'excluded_case\']]\n    cases={r[\'case\'] for r in allrows}\n    coverage={case:[(r[\'start_s\'],r[\'end_s\']) for r in allrows if r[\'case\']==case] for case in cases}\n    events=pd.DataFrame([e for e in data.events if e[\'case\'] in cases])\n    return p,events,coverage\n'
import hashlib
from datetime import datetime,timezone
OUTPUT=RUN/'event_scoring_v1'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT.mkdir(parents=True,exist_ok=False)
protocol=dict(stage='event_scoring_v1',status='synthetic_tests_passed_real_predictions_not_scored',
              sph_s=300,sop_s=1800,cooldown_s=2100,decision_bin_s=4,
              sop_boundary='(alarm+300, alarm+2100]',trigger='ready score >= threshold; repeat only after cooldown',
              threshold=None,threshold_selection='pending development-only objective and validation predictions',
              synthetic_threshold=0.5,ictal_gate=False,postictal_suppression=False,
              sensitivity_denominator='all locked eligible targets in evaluated role',
              false_alarm_definition='no annotated seizure onset in SOP and full future annotation coverage',
              rate_denominator='ready decision-bin hours with fully observed following 35 min; includes ictal/postictal and cooldown',
              warning_burden='35-minute active alerts sampled on ready/evaluable four-second decision bins',
              non_target_seizures='separate alarm outcome; not target success',
              missing_outcome='unscorable; observed positives retained and flagged if future incomplete',
              scorer_sha256=hashlib.sha256(SCORER_SOURCE.encode()).hexdigest(),
              folds_sha256=hashlib.sha256((RUN/'locked_folds.json').read_bytes()).hexdigest(),
              tests=passed_tests,real_model_metrics_computed=False)
(OUTPUT/'scoring_protocol.json').write_text(json.dumps(protocol,indent=2))
(OUTPUT/'event_scoring.py').write_text(SCORER_SOURCE)
synthetic_alarms.to_json(OUTPUT/'synthetic_alarms.json',orient='records',indent=2)
synthetic_targets.to_json(OUTPUT/'synthetic_targets.json',orient='records',indent=2)
print(json.dumps(protocol,indent=2))
print('Saved:',OUTPUT)

## Later integration with trained models — do not run on smoke previews
The full prediction export must contain **every stored window** for the chosen role, including unavailable windows:

`relative_path, index, case, decision_s, prediction_ready, score`

Use `NaN` scores for unavailable rows; preserve label -1 windows. Score means the fixed causal Top-8 score. Include a provenance JSON with `purpose="full_chronological_predictions"`, `smoke_only=false`, `outer`, `scope`, `role`, `scaler_sha256`, `folds_sha256`, and `model_checkpoint_sha256`.

With the trained-model `FoldData` instance named `data`, call `validate_full_prediction_export(data, role, predictions, provenance)` to check every file/index, availability mask, timestamps, role and hashes. It returns predictions, all seizure annotations and annotation coverage. Then call `score_cohort` with an **explicit** `ScoringPolicy(threshold=selected_threshold)`; do not use the demonstration default for actual evaluation.

The validator cannot prove how a model was trained or how a threshold was selected. Training provenance and development-only selection must also be enforced by the eventual training workflow. Score all outer folds only after model/threshold selection is locked. Confidence intervals and chance-level comparison remain additional analysis stages.

References for concepts (specific operational conventions above are explicit study choices): [Winterhalder et al., 2003](https://pubmed.ncbi.nlm.nih.gov/12791335/) separates prediction horizon and occurrence period. This notebook does not copy a published clinical operating threshold.

**Next:** specify and test the physics regularizer, then run a development-only pilot with these shared scoring rules. Before selecting a threshold, fix the false-alarm budget or another objective; no clinical acceptability threshold is assumed here.